# **Conditional Variational Autoencoder (CVAE)**

We further extend the convolutional Variational Autoencoder by incorporating conditional information in the form of cluster-based pseudo-labels, resulting in a Conditional Variational Autoencoder (CVAE). Unlike a standard VAE, the CVAE learns latent representations that are explicitly guided by high-level grouping information, enabling better disentanglement of latent factors.

The CVAE conditions both the encoder and decoder on cluster assignments obtained from unsupervised clustering of audio latent features. This encourages the model to capture meaningful intra-cluster variations while maintaining separation between different clusters. By integrating convolutional layers with conditional learning, the CVAE produces more structured and interpretable audio latent spaces, improving clustering quality and representation robustness.

In [21]:
# Import necessary libraries
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
import pandas as pd
from pathlib import Path
import os
import glob
import soundfile as sf
import librosa
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from torch.utils.data import Dataset, DataLoader

In [2]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
# Load audio files from the specified directory
audio_dir = Path("../data/audio")
audio_files = sorted(audio_dir.glob("*/*.mp3"))

print(f"Loaded {len(audio_files)} audio files")

Loaded 3554 audio files


In [4]:
# Extract Mel-spectrogram features
def load_audio(path, target_sr=22050):
    audio, sr = sf.read(path, dtype='float32')

    # Convert stereo to mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if needed (keyword arguments!)
    if sr != target_sr:
        audio = librosa.resample(
            y=audio,
            orig_sr=sr,
            target_sr=target_sr
        )

    return audio

In [5]:
# Extract Mel-spectrogram features
def extract_mel(path, n_mels=64, fixed_len=1304):  
    y, sr = librosa.load(path, sr=22050)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=512)
    mel = np.log(mel + 1e-9)
    
    # Normalize to [0,1]
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-9)
    
    # Pad or truncate to fixed_len
    if mel.shape[1] < fixed_len:
        pad_width = fixed_len - mel.shape[1]
        mel = np.pad(mel, ((0,0), (0,pad_width)), mode='constant')
    else:
        mel = mel[:, :fixed_len]

    return torch.tensor(mel, dtype=torch.float32)

In [6]:
# Test the extract_mel function
x = extract_mel(audio_files[0])
print(x.shape)

torch.Size([64, 1304])


In [8]:
# Define Dataset class for CVAE
class AudioDatasetCVAE(Dataset):
    def __init__(self, audio_files, cluster_labels, n_mels=64, fixed_len=1300):
        self.audio_files = audio_files
        self.cluster_labels = cluster_labels
        self.n_mels = n_mels
        self.fixed_len = fixed_len
        self.num_classes = len(np.unique(cluster_labels))

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        mel = extract_mel(self.audio_files[idx], self.n_mels, self.fixed_len)
        mel = mel.unsqueeze(0)  # (1, 64, 1300)

        label = self.cluster_labels[idx]
        condition = torch.zeros(self.num_classes)
        condition[label] = 1.0  # one-hot

        return mel, condition


In [10]:
# Define the Conditional Variational Autoencoder (CVAE) model
class ConvCVAE(nn.Module):
    def __init__(self, n_mels=64, fixed_len=1300, latent_dim=32, num_classes=10):
        super().__init__()
        self.num_classes = num_classes

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )

        dummy = torch.zeros(1, 1, n_mels, fixed_len)
        h = self.encoder(dummy)
        self.enc_shape = h.shape[1:]
        self.flat_dim = h.view(1, -1).size(1)

        # Condition injected here
        self.fc_mu = nn.Linear(self.flat_dim + num_classes, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim + num_classes, latent_dim)

        # Decoder
        self.fc_decode = nn.Linear(latent_dim + num_classes, self.flat_dim)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def encode(self, x, c):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        hc = torch.cat([h, c], dim=1)
        return self.fc_mu(hc), self.fc_logvar(hc)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, c):
        zc = torch.cat([z, c], dim=1)
        h = self.fc_decode(zc)
        h = h.view(h.size(0), *self.enc_shape)
        return self.decoder(h)

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, c)
        return recon, mu, logvar

In [11]:
# Define the CVAE loss function
def cvae_loss(recon_x, x, mu, logvar):
    recon = nn.functional.mse_loss(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kld

In [12]:
# Training loop for CVAE
def train_cvae(audio_files, cluster_labels, epochs=10, batch_size=8):
    dataset = AudioDatasetCVAE(audio_files, cluster_labels)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ConvCVAE(num_classes=len(np.unique(cluster_labels))).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        total_loss = 0
        for x, c in loader:
            x, c = x.to(device), c.to(device)
            optimizer.zero_grad()
            recon, mu, logvar = model(x, c)
            loss = cvae_loss(recon, x, mu, logvar)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(loader):.6f}")

    return model

In [14]:
# Example usage
model = ConvCVAE(n_mels=64, fixed_len=1300, latent_dim=32)
sum(p.numel() for p in model.parameters() if p.requires_grad)

8977025

In [15]:
# Training loop for ConvCVAE 
model = ConvCVAE(n_mels=64, fixed_len=1300, latent_dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
torch.save(model.state_dict(), "../results/models/conv_audio_cvae.pth")

In [23]:
# Load the previously saved latent vectors
z_audio = np.load("../results/z_audio.npy")
print("z_audio loaded successfully!")
print("Shape:", z_audio.shape)

z_audio loaded successfully!
Shape: (3554, 32)


In [24]:
# Perform KMeans clustering on the latent vectors
num_clusters = 10 
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(z_audio)

print("cluster_labels shape:", cluster_labels.shape)
print("unique clusters:", np.unique(cluster_labels))

cluster_labels shape: (3554,)
unique clusters: [0 1 2 3 4 5 6 7 8 9]


In [ ]:
# Extract latent vectors using the trained CVAE model
model.eval()
latent_vectors = []

# Dataset and DataLoader
dataset = AudioDatasetCVAE(
    audio_files=audio_files,
    cluster_labels=cluster_labels,
    n_mels=64,
    fixed_len=1304
)

loader = DataLoader(dataset, batch_size=1, shuffle=False)

# Extract latent vectors
with torch.no_grad():
    for x, c in loader:
        x = x.to(device)   
        c = c.to(device)   

        mu, logvar = model.encode(x, c)
        z = model.reparameterize(mu, logvar)

        latent_vectors.append(z.cpu().numpy())

# Convert to numpy array
z_audio_CVAE = np.vstack(latent_vectors)
print("Audio latent vectors shape:", z_audio_CVAE.shape)

Audio latent vectors shape: (3554, 32)


In [26]:
# Save latent vectors
np.save("../results/z_audio_CVAE.npy", z_audio_CVAE)
print("z_audio_CVAE saved successfully!")

z_audio_CVAE saved successfully!
